### This EXTRACT_CORPUS notebook 

- Extracts the works for each journal

- Load the works into the database after  
    - Filters works into a flat table (work_id, doi, source, host, citation_count etc)  
    - Flattens the authorships table for each work (author_id, institution_id etc)  
    - Filters the reference list to make the cited table (When inverted these are the endogenous citations)  

Note that at this stage the works have been filtered by publication date and type (articles, etc)


In [7]:
%run common_setup.ipynb

In [8]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return

    def extract_works_by_journal(self):

        sql = """
                -- ETL TO EXTRACT works FOR journal list
                -- ======================================
                CREATE OR REPLACE TABLE project.raw AS
                SELECT -- DISTINCT ON (doi)
                        w.*,
                        lower(title) AS lower_title,
                        IF (first_page = last_page, 0, try_cast(last_page AS INT) - try_cast(first_page AS INT)) AS page_count
                FROM project.sources_matched sm
                LEFT JOIN works.works w
                USING (source_id)
                WHERE publication_year > 2009 AND publication_year < 2025
                        AND (is_retracted = false OR is_retracted IS NULL)
                        AND is_paratext = false
                        -- AND (first_page IS NULL OR page_count > 1)
                        -- AND doi IS NOT NULL
                        -- AND referenced_works_count != 0
                        AND contains(lower_title, 'editor') = false 
                        AND contains(lower_title, 'issue information') = false
                        AND contains(lower_title, 'index') = false
                        AND contains(lower_title, 'book review') = false
                        AND contains(lower_title, 'isbn') = false
                        AND contains(lower_title, 'calendar of events') = false
                        AND contains(lower_title, 'notes on contributors') = false
                        AND regexp_matches(lower_title, '^announcements$') = false
                        AND regexp_matches(lower_title, '^acknowledgement') = false
                        AND regexp_matches(lower_title, '^vol[.u ] ') = false
                        AND regexp_matches(lower_title, '^focus on authors$') = false
                        AND regexp_matches(lower_title, '^publications received$') = false
                        AND list_contains(['article', 'review', 'letter'], w."type") = true
                -- ORDER BY page_count
        """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) FROM project.raw").show()
        self.db.sql("SELECT * FROM project.raw").show()
        return
 
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return

In [9]:
class ExtractAuthorshipsReferencesTopics(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def authorships_etl(self):
        print("authorships")
        sql = """
            -- ETL TO EXTRACT authorships FOR project
            -- ======================================
            CREATE OR REPLACE TABLE project.authorships AS
                SELECT a.* 
                    FROM project.raw
                    INNER JOIN works.authorships a
                    ON id = work_id
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(DISTINCT work_id), count(DISTINCT author_id), count(DISTINCT institution_id) FROM project.authorships").show()
        return
    
    def references_etl(self):
        print("references")
        sql = """
            -- ETL FOR endogenous citer_cited
            -- =============================================
            CREATE OR REPLACE TABLE project.citer_cited AS
                WITH
                citer_cited_CTE AS
                    (SELECT id AS citer_id,
                            publication_year AS citer_year,
                            unnest(referenced_works) AS cited_id
                        FROM project.raw -- ALL citers ARE FROM raw
                    ),
                citer_cited_filtered_CTE AS
                    (SELECT DISTINCT citer_id,
                            citer_year,
                            cited_id,
                            publication_year AS cited_year
                        FROM citer_cited_CTE
                        INNER JOIN project.raw -- ALL cited are from raw
                        ON cited_id = id
                    )
            
            SELECT *,
                    cited_year - citer_year - 1 AS delta_t
                FROM citer_cited_filtered_CTE
                WHERE delta_t <= 0
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*), count(DISTINCT citer_id), count(DISTINCT cited_id) FROM project.citer_cited").show()
        return
    
    def topics_etl(self):
        print("topics")
        sql = """
            -- ETL FOR topics MATCHES raw.work_id to works.topics
            -- ==================================================
            CREATE OR REPLACE TABLE project.topics AS
            SELECT DISTINCT r.id AS work_id,
                    topic_score,
                    topic_id,
                    display_name AS topic_name,
                    subfield.id AS subfield_id,
                    subfield.display_name AS subfield_name,
                    field.id AS field_id,
                    field.display_name AS field_name,
                    domain.id AS domain_id,
                    domain.display_name AS dmoain_name
                FROM project.raw r
                INNER JOIN works.topics w
                ON r.id = w.work_id
                LEFT JOIN topics.topics t
                ON t.id = topic_id
            """
        self.db.sql(sql)
        self.db.sql("""SELECT count(*), 
                                count(DISTINCT work_id), count(DISTINCT topic_id), count(DISTINCT subfield_id),
                                count(DISTINCT field_id), count(DISTINCT domain_id)  FROM project.topics""").show()
        return

In [10]:
class  ExtractAuthors(SetUp):

    def __init__(self):
        super().__init__()
        self._prepare_cited_by()
        return

    def _prepare_cited_by(self):
        sql = """ 
            -- ETL TO compute ENDOGENOUS cited_by_count AND cited_by_self_reference 
            -- ====================================================================
            CREATE OR REPLACE TABLE memory.cited_by_count_endogenous AS
            WITH
                author_list_CTE AS
                (
                SELECT DISTINCT work_id,
                        list(DISTINCT author_id) AS author_list
                        FROM project.authorships
                        GROUP BY work_id 
                ),
                common_authors_CTE AS
                (SELECT DISTINCT citer_id, cited_id,
                        IF(len(list_intersect(a1_list, a2.author_list)) > 1, 1, 0) AS self_reference
                    FROM 
                        (SELECT citer_id, cited_id, 
                                a1.author_list AS a1_list,
                        FROM project.citer_cited
                        LEFT JOIN author_list_CTE a1
                        ON citer_id = a1.work_id
                        )
                    LEFT JOIN author_list_CTE a2
                    ON cited_id = a2.work_id
                    ),
                flag_self_references_CTE AS
                    (SELECT DISTINCT citer_id, cited_id, self_reference,
                            a3.author_id AS cited_author
                    FROM  common_authors_CTE
                    LEFT JOIN project.authorships a3
                    ON a3.work_id = cited_id
                    WHERE author_id NOT NULL
                    )
            
            SELECT cited_author AS author_id,
                    count(citer_id) AS cited_by_count_endogenous,
                    SUM(self_reference) AS cited_by_self_referenced,
                    SUM(self_reference)/count(citer_id) AS self_reference_rate
                FROM flag_self_references_CTE
                GROUP BY cited_author
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.cited_by_count_endogenous").show()
        return

    def extract_authors(self):
        # Extract OA authors for the journal set
        sql = """
            -- ETL authors FROM raw.authorships JOIN authors.authors INSERT endogenous cites and self references
            -- =================================================================================================
            CREATE OR REPLACE TABLE project.authors AS
                WITH
                    author_details_CTE AS
                    (SELECT DISTINCT ON (a.author_id)
                        a.author_id,
                        a.author_name,
                        aa.orcid,
                        display_name_alternatives,
                        count(DISTINCT a.work_id) AS works_count_endogenous,
                        summary_stats.works_count AS works_count, 
                        summary_stats.cited_by_count AS cited_by_count, 
                        summary_stats.h_index AS h_index,
                        summary_stats."2yr_h_index" AS h_index_2yr,
                    FROM project.authorships a
                    INNER JOIN authors.authors aa
                    ON aa.id = a.author_id
                    GROUP BY ALL
                    )

                SELECT DISTINCT *
                FROM author_details_CTE d
                    LEFT JOIN memory.cited_by_count_endogenous c
                    USING (author_id)
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.authors").show()
        return

    def prepare_normalised_alt_names(self):
        sql = """
            -- ETL author_alt_names from raw.authors
            -- =====================================
            CREATE OR REPLACE TABLE project.authors_alt AS
                SELECT author_id,
                        author_name,
                        normalise_name(alt_name)[1] AS first,
                        normalise_name(alt_name)[2] AS middle,
                        normalise_name(alt_name)[3] AS last,
                        normalise_name(alt_name)[4] AS fullname
                    FROM 
                        (SELECT author_id,
                                author_name,
                                unnest(display_name_alternatives) AS alt_name
                            FROM project.authors
                        )
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.authors_alt").show()
        return
    
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return         

In [11]:
def main():

    # jetl = ArticlesETL()
    # jetl.extract_works_by_journal()
    # print(jetl.db.sql("DESCRIBE TABLE project.raw").df())
    # jetl.db.sql("SELECT * FROM project.raw").show()
    # sql = """SELECT count(DISTINCT source_id) FROM project.raw"""
    # jetl.db.sql(sql).show()
    # # # jetl.duplicate_db_as_backup()
    # jetl.db.close()

    # ea = ExtractAuthorshipsReferencesTopics()
    # ea.authorships_etl()
    # ea.references_etl()
    # ea.topics_etl()
    # ea.db.close()

    eauthors = ExtractAuthors()
    eauthors.extract_authors()
    # eauthors.prepare_normalised_alt_names()
    # # eauthors.duplicate_db_as_backup()
    eauthors.db.close()

In [12]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────┬───────────┐
│   database   │ schema  │         name         │     column_names     │           column_types            │ temporary │
│   varchar    │ varchar │       varchar        │      varchar[]       │             varchar[]             │  boolean  │
├──────────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────┼───────────┤
│ authors      │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ institutions │ main    │ institutions         │ [id, ror, display_…  │ [VARCHAR, VARCHAR, VARCHAR, VAR…  │ false     │
│ institutions │ main    │ ror                  │ [name, institution…  │ [VARCHAR, VARCHAR]                │ false     │
│ project      │ main    │ author_citation_re…  │ [dupes, author_id,…  │ [BIGINT, VARCHAR, VARCHAR, VARC…  │ false     │
│ project      │ main    │ autho